<a href="https://colab.research.google.com/github/dianakim0105/Score-based-Modelling-Diffusion-Modelling/blob/main/VE_Windkessel_Bayesian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Bayesian score matching + PC sampling for a flow-driven 3-element Windkessel model.

  phi = (C, Rd) inferred; Rc fixed at RC_FIXED
  x | theta ~ N(simulator(theta), sigma_lik^2 I)   [full pressure vector, mmHg]

Diffusion in standardized log-space on phi. Network sees z-scored PCA scores of x
(fit on prior-predictive pressures; smallest k with >= 99% variance).
Likelihood and IS reference use raw mmHg observations.
"""

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


# ============================================================
# Fixed hemodynamics — not inferred
# Units: t [s], Q [ml/s], P [mmHg], R [mmHg·s/ml], C [ml/mmHg]
# ============================================================
T_SYS = 0.3
T_CYCLE = 0.8
Q_MAX = 400.0  # peak inlet flow during systole
P_ART_INIT = 15.0
SIM_DT = 0.002

RC_FIXED = 0.25  # mmHg·s/ml — proximal resistance fixed (not inferred)

# Prior bounds for inferred parameters phi = (C, Rd)
WK_BOUNDS_INFER = torch.tensor(
    [
        [0.02, 0.25],   # C  (ml/mmHg)
        [0.20, 1.50],   # Rd (mmHg·s/ml)
    ],
    dtype=torch.float32,
)

OBS_SUBSAMPLE = 10
PCA_VARIANCE_THRESHOLD = 0.99


# ============================================================
# 1) Prior + Windkessel simulator + Gaussian likelihood
# ============================================================
def _time_grid(device="cpu"):
    t_end = 2.0 * T_CYCLE
    return torch.arange(0.0, t_end + SIM_DT, SIM_DT, device=device, dtype=torch.float32)


def inlet_flow(t):
    """
    Prescribed cardiac inflow (ml/s).
    Systole: Q_in(t) = Q_max * sin^2(pi * t / T_sys); diastole: Q_in = 0.
    """
    tmod = (t % T_CYCLE).item()
    if tmod < T_SYS:
        s = math.sin(math.pi * tmod / T_SYS)
        return Q_MAX * s * s
    return 0.0


def simulate_pressure_trajectory(theta, device="cpu"):
    """
    Flow-driven 3-element Windkessel (RCR).

    State P: pressure at the arterial compliance (distal side of Rc).
        C * dP/dt = Q_in(t) - P / Rd

    Observed aortic pressure (proximal, across characteristic resistance Rc):
        P_aorta = P + Rc * Q_in(t)

    theta: (3,) or (B, 3) with columns [Rc, C, Rd]
    returns P_aorta: (n_t,) or (B, n_t)
    """
    single = theta.ndim == 1
    if single:
        theta = theta.unsqueeze(0)

    theta = theta.to(device=device, dtype=torch.float32)
    B = theta.shape[0]
    Rc, C, Rd = theta[:, 0], theta[:, 1], theta[:, 2]

    times = _time_grid(device)
    n_t = times.numel()
    P = torch.full((B,), P_ART_INIT, device=device, dtype=torch.float32)
    traj = torch.empty(B, n_t, device=device, dtype=torch.float32)

    q0 = inlet_flow(times[0])
    traj[:, 0] = P + Rc * q0

    for i in range(1, n_t):
        t = times[i]
        q_in = inlet_flow(t)
        dP = (q_in - P / Rd) / C
        P = P + dP * SIM_DT
        P_aorta = P + Rc * q_in
        traj[:, i] = P_aorta

    return traj.squeeze(0) if single else traj


def trajectory_to_observations(traj):
    """Subsample pressure trajectory -> observation vector."""
    if traj.ndim == 1:
        return traj[::OBS_SUBSAMPLE]
    return traj[:, ::OBS_SUBSAMPLE]


def simulator_features(theta, device="cpu"):
    """theta (B,3) -> noiseless features (B, d_obs)."""
    traj = simulate_pressure_trajectory(theta, device=device)
    return trajectory_to_observations(traj)


def assemble_theta(phi, device="cpu", rc=None):
    """phi (B,2) or (2,) with [C, Rd] -> theta (B,3) or (3,) with [Rc, C, Rd]."""
    if rc is None:
        rc = RC_FIXED
    if phi.ndim == 1:
        return torch.tensor([rc, phi[0], phi[1]], device=device, dtype=torch.float32)
    B = phi.shape[0]
    rc_t = torch.full((B,), float(rc), device=device, dtype=torch.float32)
    return torch.stack([rc_t, phi[:, 0], phi[:, 1]], dim=1)


def sample_prior_uniform_phi(n_samples, bounds=None, device="cpu"):
    """Uniform prior on inferrable phi = (C, Rd)."""
    if bounds is None:
        bounds = WK_BOUNDS_INFER
    bounds = bounds.to(device)
    lo, hi = bounds[:, 0], bounds[:, 1]
    u = torch.rand(n_samples, 2, device=device, dtype=torch.float32)
    return lo + u * (hi - lo)


def log_likelihood_gaussian(x, theta, sigma_lik, device="cpu"):
    """
    x:     (d_obs,) or (B, d_obs) observed
    theta: (B, 3)
    returns log p(x | theta), shape (B,)
    """
    pred = simulator_features(theta, device=device)
    if x.ndim == 1:
        x = x.unsqueeze(0).expand(theta.shape[0], -1)
    resid = x - pred
    d = resid.shape[1]
    var = float(sigma_lik) ** 2
    return -0.5 * d * math.log(2 * math.pi * var) - 0.5 * (resid ** 2).sum(dim=1) / var


def simulate_gaussian_likelihood(theta, sigma_lik, device="cpu"):
    """x | theta ~ N(sim(theta), sigma_lik^2 I). theta: (B,3)."""
    mean = simulator_features(theta, device=device)
    return mean + float(sigma_lik) * torch.randn_like(mean)


def sample_joint_prior_simulator(n_samples, sigma_lik, bounds=None, device="cpu"):
    phi = sample_prior_uniform_phi(n_samples, bounds=bounds, device=device)
    theta = assemble_theta(phi, device=device)
    x = simulate_gaussian_likelihood(theta, sigma_lik, device=device)
    return theta, phi, x


# ============================================================
# Log-standardize parameters; z-score observations (network only)
# ============================================================
class LogStandardizeTransform:
    """Physical phi > 0 -> u = (log(phi) - mu) / std from uniform prior on log-coords."""

    def __init__(self, bounds=None, device="cpu"):
        if bounds is None:
            bounds = WK_BOUNDS_INFER
        bounds = bounds.to(device=device, dtype=torch.float32)
        log_lo = torch.log(bounds[:, 0])
        log_hi = torch.log(bounds[:, 1])
        self.mu = (log_lo + log_hi) / 2.0
        self.std = (log_hi - log_lo) / math.sqrt(12.0)

    def to(self, device):
        self.mu = self.mu.to(device)
        self.std = self.std.to(device)
        return self

    def to_latent(self, phi_phys):
        return (torch.log(phi_phys) - self.mu) / self.std

    def to_physical(self, u):
        return torch.exp(self.mu + self.std * u)


class ObservationPCA:
    """
    PCA of prior-predictive pressure vectors for network input.
    encode() returns z-scored PC scores; likelihood stays in mmHg.
    """

    def __init__(self, mean_x, components, score_mean, score_std, variance_explained):
        self.mean_x = mean_x
        self.components = components
        self.score_mean = score_mean
        self.score_std = score_std
        self.n_components = components.shape[0]
        self.variance_explained = variance_explained

    def to(self, device):
        self.mean_x = self.mean_x.to(device=device, dtype=torch.float32)
        self.components = self.components.to(device=device, dtype=torch.float32)
        self.score_mean = self.score_mean.to(device=device, dtype=torch.float32)
        self.score_std = self.score_std.to(device=device, dtype=torch.float32)
        return self

    def project(self, x):
        """Raw PC scores (no z-score). x: (d,) or (B, d)."""
        if x.ndim == 1:
            xc = x - self.mean_x
            return xc @ self.components.T
        xc = x - self.mean_x
        return xc @ self.components.T

    def encode(self, x):
        """Z-scored PC scores for the score network."""
        scores = self.project(x)
        return (scores - self.score_mean) / self.score_std

    @classmethod
    @torch.no_grad()
    def fit_from_prior_predictive(
        cls,
        sigma_lik,
        variance_threshold=PCA_VARIANCE_THRESHOLD,
        n_fit=4000,
        bounds=None,
        device="cpu",
        score_std_floor=1e-3,
    ):
        _, _, x = sample_joint_prior_simulator(n_fit, sigma_lik, bounds=bounds, device=device)
        X = x.cpu().numpy()
        mean_np = X.mean(axis=0)
        Xc = X - mean_np
        _, s, vt = np.linalg.svd(Xc, full_matrices=False)
        var = (s ** 2) / max(n_fit - 1, 1)
        cumvar = np.cumsum(var) / (var.sum() + 1e-30)
        n_pc = int(np.searchsorted(cumvar, variance_threshold) + 1)
        n_pc = max(1, min(n_pc, X.shape[1]))
        components = vt[:n_pc]
        scores = Xc @ components.T
        score_mean = scores.mean(axis=0)
        score_std = np.maximum(scores.std(axis=0), score_std_floor)
        var_expl = float(cumvar[n_pc - 1])
        return cls(
            mean_x=torch.tensor(mean_np, dtype=torch.float32),
            components=torch.tensor(components, dtype=torch.float32),
            score_mean=torch.tensor(score_mean, dtype=torch.float32),
            score_std=torch.tensor(score_std, dtype=torch.float32),
            variance_explained=var_expl,
        ).to(device)


def build_transform(bounds=None, device="cpu"):
    return LogStandardizeTransform(bounds=bounds, device=device)


# ============================================================
# 2) Reference posterior via importance sampling (uniform prior)
# ============================================================
@torch.no_grad()
def sample_posterior_importance(
    x_obs,
    sigma_lik,
    n_samples,
    n_candidates=200_000,
    bounds=None,
    device="cpu",
):
    """Resample uniform prior candidates weighted by likelihood."""
    if bounds is None:
        bounds = WK_BOUNDS_INFER
    x_obs = torch.as_tensor(x_obs, dtype=torch.float32, device=device)
    phi_cand = sample_prior_uniform_phi(n_candidates, bounds=bounds, device=device)
    candidates = assemble_theta(phi_cand, device=device)
    logw = log_likelihood_gaussian(x_obs, candidates, sigma_lik, device=device)
    logw = logw - logw.max()
    w = torch.exp(logw)
    w = w / (w.sum() + 1e-30)
    idx = torch.multinomial(w, n_samples, replacement=True)
    return candidates[idx]


@torch.no_grad()
def exact_noisy_posterior_score_latent(
    z,
    x_obs,
    sigma_lik,
    sigma_ve,
    transform,
    bounds=None,
    mc_samples=50_000,
    device="cpu",
):
    """
    Noisy posterior score in standardized log-space u.
    IS samples theta ~ p(theta|x); VE kernel weights on u = T(theta) only.
    """
    z = z.to(device)
    theta_s = sample_posterior_importance(
        x_obs, sigma_lik, mc_samples, bounds=bounds, device=device
    )
    phi_s = theta_s[:, 1:3]
    u_s = transform.to_latent(phi_s)

    z_e = z.unsqueeze(1)
    u_e = u_s.unsqueeze(0)
    log_w = -0.5 * ((z_e - u_e) ** 2).sum(dim=2) / (float(sigma_ve) ** 2)
    log_w = log_w - log_w.max(dim=1, keepdim=True).values
    w = torch.exp(log_w)
    denom = w.sum(dim=1, keepdim=True) + 1e-12
    e_u = (w.unsqueeze(2) * u_e).sum(dim=1) / denom
    return -(z - e_u) / (float(sigma_ve) ** 2)


# ============================================================
# 3) Score network
# ============================================================
class PosteriorScoreNet(nn.Module):
    """MLP: [u_t, PC scores (z-scored), log sigma] -> score in latent phi-space (2D)."""

    def __init__(self, n_pc, param_dim=2, hidden_dim=128, n_hidden_layers=2):
        super().__init__()
        in_dim = param_dim + n_pc + 1
        layers = []
        d_in = in_dim
        for _ in range(n_hidden_layers):
            layers.extend([nn.Linear(d_in, hidden_dim), nn.Tanh()])
            d_in = hidden_dim
        layers.append(nn.Linear(hidden_dim, param_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, u_t, x_pc, sigma):
        if sigma.ndim == 1:
            sigma = sigma.unsqueeze(1)
        log_sigma = torch.log(sigma)
        inp = torch.cat([u_t, x_pc, log_sigma], dim=1)
        return self.net(inp)


# ============================================================
# 4) DSM training
# ============================================================
def train_bayesian_dsm(
    model,
    sigma_lik,
    sigmas,
    transform,
    obs_pca,
    bounds=None,
    steps=8000,
    batch_size=64,
    lr=1e-3,
    sigma_weight=True,
    device="cpu",
):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sigmas_t = torch.tensor(sigmas, dtype=torch.float32, device=device)

    for step in range(1, steps + 1):
        _, phi0, x = sample_joint_prior_simulator(
            batch_size, sigma_lik, bounds=bounds, device=device
        )
        u0 = transform.to_latent(phi0)
        x_net = obs_pca.encode(x)

        idx = torch.randint(0, sigmas_t.numel(), (batch_size,), device=device)
        sigma = sigmas_t[idx]

        eps = torch.randn_like(u0)
        u_t = u0 + sigma.unsqueeze(1) * eps

        pred = model(u_t, x_net, sigma)
        target = -eps / sigma.unsqueeze(1)

        if sigma_weight:
            w = (sigma ** 2).unsqueeze(1)
            loss = ((pred - target) ** 2 * w).mean()
        else:
            loss = ((pred - target) ** 2).mean()

        opt.zero_grad()
        loss.backward()
        opt.step()

        if step % 500 == 0:
            print(f"step {step:5d} | loss {loss.item():.6f}")

    return model


# ============================================================
# 5) Score diagnostics
# ============================================================
@torch.no_grad()
def eval_score_diagnostics(
    model,
    x_obs,
    sigma_lik,
    sigmas,
    transform,
    obs_pca,
    bounds=None,
    n_test=256,
    mc_samples=20_000,
    device="cpu",
):
    model.eval()
    x_obs_t = torch.as_tensor(x_obs, dtype=torch.float32, device=device)
    x_obs_net = obs_pca.encode(x_obs_t)

    print("\nScore diagnostics vs IS reference (standardized log-space)")
    print(f"{'sigma':>8}  {'MSE':>10}  {'cosine':>10}")

    for sigma in sigmas:
        sigma = float(sigma)
        theta0 = sample_posterior_importance(
            x_obs_t, sigma_lik, n_test, bounds=bounds, device=device
        )
        phi0 = theta0[:, 1:3]
        u0 = transform.to_latent(phi0)
        x = x_obs_net.unsqueeze(0).expand(n_test, -1)
        eps = torch.randn_like(u0)
        z = u0 + sigma * eps
        sigma_batch = torch.full((n_test,), sigma, dtype=torch.float32, device=device)

        pred = model(z, x, sigma_batch)
        true = exact_noisy_posterior_score_latent(
            z, x_obs_t, sigma_lik, sigma, transform, bounds=bounds,
            mc_samples=mc_samples, device=device,
        )

        mse = ((pred - true) ** 2).mean().item()
        cos = F.cosine_similarity(pred, true, dim=1).mean().item()
        print(f"{sigma:8.4f}  {mse:10.2f}  {cos:10.4f}")


# ============================================================
# 6) Predictor–Corrector posterior sampler
# ============================================================
@torch.no_grad()
def sample_posterior_pc(
    model,
    x_obs,
    sigma_lik,
    sigmas,
    transform,
    obs_pca,
    bounds=None,
    n_samples=2000,
    n_corrector_steps=2,
    step_scale=1e-5,
    device="cpu",
    return_latent=False,
):
    """
    PC diffusion in standardized log-space u on phi = (C, Rd).
    Init: u = T(phi_prior) + sigma_max * eps.
    """
    model.eval()
    x_obs_t = torch.as_tensor(x_obs, dtype=torch.float32, device=device)
    x_batch = obs_pca.encode(x_obs_t).unsqueeze(0).expand(n_samples, -1)

    sigmas_t = torch.tensor(sigmas, dtype=torch.float32, device=device)
    sigma_max = sigmas_t.max().item()
    sigma_min = sigmas_t.min().item()

    phi_prior = sample_prior_uniform_phi(n_samples, bounds=bounds, device=device)
    u = transform.to_latent(phi_prior) + sigma_max * torch.randn(n_samples, 2, device=device)

    for i in range(len(sigmas_t) - 1):
        sigma_i = float(sigmas_t[i].item())
        sigma_next = float(sigmas_t[i + 1].item())
        sigma_batch_i = torch.full((n_samples,), sigma_i, device=device)

        score = model(u, x_batch, sigma_batch_i)
        delta = sigma_i**2 - sigma_next**2
        u = u + delta * score + math.sqrt(max(delta, 0.0)) * torch.randn_like(u)

        sigma_batch_next = torch.full((n_samples,), sigma_next, device=device)
        for _ in range(n_corrector_steps):
            score = model(u, x_batch, sigma_batch_next)
            eta = step_scale * ((sigma_next / sigma_min) ** 2)
            u = u + eta * score + math.sqrt(2.0 * eta) * torch.randn_like(u)

    if return_latent:
        return u
    phi = transform.to_physical(u)
    return assemble_theta(phi, device=device)


# ============================================================
# 7) Plots
# ============================================================
@torch.no_grad()
def plot_posterior_marginals(
    samples_phys,
    x_obs,
    sigma_lik,
    transform,
    bounds=None,
    device="cpu",
    bins=40,
):
    ref_phys = sample_posterior_importance(
        x_obs, sigma_lik, 5000, bounds=bounds, device=device
    )
    ref_phi = ref_phys[:, 1:3]
    ref_latent = transform.to_latent(ref_phi)
    bounds_np = (bounds if bounds is not None else WK_BOUNDS_INFER).cpu().numpy()
    names = ["C", "Rd"]
    samp_phi = samples_phys[:, 1:3]
    samp_latent = transform.to_latent(samp_phi)

    fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
    for j, ax in enumerate(axes):
        ax.hist(samp_latent[:, j].cpu().numpy(), bins=bins, density=True, alpha=0.5, label="PC")
        ax.hist(ref_latent[:, j].cpu().numpy(), bins=bins, density=True, alpha=0.4,
                histtype="step", linewidth=2, label="IS ref")
        ax.set_xlabel(f"{names[j]} (std. log)")
        ax.set_ylabel("Density")
    axes[0].legend()
    fig.suptitle(f"Posterior marginals — latent (Rc fixed at {RC_FIXED})")
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
    for j, ax in enumerate(axes):
        ax.hist(samp_phi[:, j].cpu().numpy(), bins=bins, density=True, alpha=0.5, label="PC")
        ax.hist(ref_phi[:, j].cpu().numpy(), bins=bins, density=True, alpha=0.4,
                histtype="step", linewidth=2, label="IS ref")
        ax.set_xlim(bounds_np[j, 0], bounds_np[j, 1])
        ax.set_xlabel(names[j])
        ax.set_ylabel("Density")
    axes[0].legend()
    fig.suptitle(f"Posterior marginals — physical (Rc fixed at {RC_FIXED})")
    plt.tight_layout()
    plt.show()


@torch.no_grad()
def plot_pressure_fit(samples, x_obs, sigma_lik, theta_true=None, device="cpu"):
    """Observed pressures vs IS-MAP trajectory."""
    x_obs_t = torch.as_tensor(x_obs, device=device)
    times = _time_grid(device)
    t_obs = times[::OBS_SUBSAMPLE].cpu().numpy()

    if theta_true is not None:
        traj_true = simulate_pressure_trajectory(
            torch.as_tensor(theta_true, device=device), device=device
        )
        pred_true = trajectory_to_observations(traj_true).cpu().numpy()
    else:
        pred_true = None

    cand = sample_posterior_importance(x_obs_t, sigma_lik, 500, device=device)
    ll = log_likelihood_gaussian(x_obs_t, cand, sigma_lik, device=device)
    theta_map = cand[ll.argmax()]
    traj_map = simulate_pressure_trajectory(theta_map, device=device)
    t_full = times.cpu().numpy()
    p_map = traj_map.cpu().numpy()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(t_full, p_map, label="IS-MAP trajectory", lw=2)
    ax.scatter(t_obs, x_obs_t.cpu().numpy(), s=12, c="crimson", zorder=5, label="Observations")
    if pred_true is not None:
        ax.scatter(t_obs, pred_true, s=12, facecolors="none", edgecolors="black",
                   label="True simulator at obs times")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("P_art (mmHg)")
    ax.set_title("Arterial pressure: data vs posterior MAP")
    ax.legend()
    plt.tight_layout()
    plt.show()


# ============================================================
# 8) Main experiment
# ============================================================
def run_windkessel_bayesian_experiment():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0)
    np.random.seed(0)

    bounds = WK_BOUNDS_INFER.to(device)
    transform = build_transform(bounds=bounds, device=device)
    sigma_lik = 0.6  # mmHg

    print(f"Fixed Rc = {RC_FIXED}; inferring phi = (C, Rd)")
    print("Log-standardize (uniform prior on log-coords for C, Rd):")
    print("  mu_log:", transform.mu.cpu().tolist())
    print("  std_log:", transform.std.cpu().tolist())

    phi_true = torch.tensor([0.08, 0.60], device=device)
    theta_true = assemble_theta(phi_true, device=device)
    print("True parameters [Rc, C, Rd]:", theta_true.cpu().tolist())

    x_clean = simulator_features(theta_true.unsqueeze(0), device=device).squeeze(0)
    x_obs = x_clean + sigma_lik * torch.randn_like(x_clean)
    d_obs = x_obs.numel()
    print(f"Observation dim (mmHg, likelihood): {d_obs}")

    obs_pca = ObservationPCA.fit_from_prior_predictive(
        sigma_lik, n_fit=4000, bounds=bounds, device=device
    )
    print(
        f"Observation PCA (prior predictive): {obs_pca.n_components} PCs "
        f"({100 * obs_pca.variance_explained:.2f}% variance)"
    )

    sigmas = np.geomspace(0.15, 0.01, 15).astype(np.float32)

    model = PosteriorScoreNet(
        n_pc=obs_pca.n_components, param_dim=2, hidden_dim=128, n_hidden_layers=2
    ).to(device)

    train_bayesian_dsm(
        model=model,
        sigma_lik=sigma_lik,
        sigmas=sigmas,
        transform=transform,
        obs_pca=obs_pca,
        bounds=bounds,
        steps=8000,
        batch_size=64,
        lr=1e-3,
        device=device,
    )

    eval_score_diagnostics(
        model=model,
        x_obs=x_obs,
        sigma_lik=sigma_lik,
        sigmas=[0.15, 0.08, 0.04, 0.02],
        transform=transform,
        obs_pca=obs_pca,
        bounds=bounds,
        n_test=256,
        device=device,
    )

    posterior_latent = sample_posterior_pc(
        model=model,
        x_obs=x_obs,
        sigma_lik=sigma_lik,
        sigmas=sigmas,
        transform=transform,
        obs_pca=obs_pca,
        bounds=bounds,
        n_samples=2000,
        device=device,
        return_latent=True,
    )
    posterior_phys = assemble_theta(transform.to_physical(posterior_latent), device=device)

    plot_posterior_marginals(
        posterior_phys,
        x_obs,
        sigma_lik,
        transform,
        bounds=bounds,
        device=device,
    )
    plot_pressure_fit(
        posterior_phys, x_obs, sigma_lik,
        theta_true=theta_true.cpu(), device=device,
    )

    return model, x_obs, posterior_phys, posterior_latent, transform, obs_pca


if __name__ == "__main__":
    run_windkessel_bayesian_experiment()


Fixed Rc = 0.25; inferring phi = (C, Rd)
Log-standardize (uniform prior on log-coords for C, Rd):
  mu_log: [-2.6491587162017822, -0.6019864082336426]
  std_log: [0.7291150689125061, 0.5816524028778076]
True parameters [Rc, C, Rd]: [0.25, 0.07999999821186066, 0.6000000238418579]
Observation dim (mmHg, likelihood): 81
Observation PCA (prior predictive): 3 PCs (99.78% variance)
step   500 | loss 0.958943
step  1000 | loss 0.688841
step  1500 | loss 0.623276
step  2000 | loss 0.577909
step  2500 | loss 0.772068
step  3000 | loss 0.642427
step  3500 | loss 0.704999
step  4000 | loss 0.477522
step  4500 | loss 0.569504
step  5000 | loss 0.479622
step  5500 | loss 0.443459
